# AI Warp Tool API

Using the proxy server endpoints to route requests to local or cloud backends.

In [8]:
import requests

WARP_URL = "http://localhost:11435"

def chat(model, messages, stream=False):
    r = requests.post(f"{WARP_URL}/api/chat", json={
        "model": model,
        "messages": messages,
        "stream": stream,
    }, stream=stream)
    print(r.status_code)
    if r.ok:
        if stream:
            for line in r.iter_lines():
                if line:
                    print(line)
        else:
            # print(r.json()["message"]["content"])
            print(print(r.json()))

## Version

In [17]:
r = requests.get(f"{WARP_URL}/api/version", params={"source": "local"})
print("Local:", r.status_code, r.text[:100])

r = requests.get(f"{WARP_URL}/api/version", params={"source": "cloud"})
print("Cloud:", r.status_code, r.text[:100])

Local: 200 {"version":"0.32.1"}
Cloud: 500 {"error":"Cloud base URL not configured. Set WARP_CLOUD_OLLAMA_BASE_URL."}


## Tags

In [18]:
r = requests.get(f"{WARP_URL}/api/tags", params={"source": "local"})
print("Local:", r.status_code)
if r.ok:
    print([m["name"] for m in r.json().get("models", [])])

r = requests.get(f"{WARP_URL}/api/tags", params={"source": "cloud"})
print("Cloud:", r.status_code)
if r.ok:
    print([m["name"] for m in r.json().get("models", [])])

Local: 200
['gemma4:31b-cloud']
Cloud: 500


## Chat — local model

In [4]:
chat("gemma4:e4b", [{"role": "user", "content": "hello"}])

200
Hello! How can I help you today? 😊


## Chat — cloud model via `-cloud` suffix

In [6]:
r = requests.post(f"{WARP_URL}/api/chat", json={
    "model": "gemma4:31b-cloud",
    "messages": [{"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
                 {"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["message"]["content"])

200
Ahoy there, matey! Avast! What brings a landlubber like ye to these salty waters? Speak up, or it's the plank for ye! 🏴‍☠️🦜


## Chat — cloud model via `?source=cloud` param

In [16]:
r = requests.post(f"{WARP_URL}/api/chat?source=cloud", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["message"]["content"])

404


## Chat with system prompt

In [6]:
chat("gemma4:12b-gpu-120k", [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Trong tiếng Việt chữ gì thêm dấu biến nam thành nữ?"},
])

200
Câu hỏi của bạn là một câu đố chữ hoặc mẹo khá phổ biến trong tiếng Việt. Có hai cách hiểu cho câu hỏi này tùy vào việc đây là một câu đố vui hay một câu đố mẹo:

1.  **Nếu là câu đố mẹo (Trick question):**
    Câu trả lời thường là **"Chữ Nữ"**.
    *Giải thích:* Đây là kiểu trả lời "lái" của người đặt câu hỏi. Họ không yêu cầu bạn biến một chữ đang là "nam" thành "nữ", mà họ chỉ muốn bạn nghĩ đến từ "Nữ". Dù có thêm dấu hay không, chữ đó vẫn mang nghĩa là nữ giới.

2.  **Nếu bạn đang nhớ nhầm với một câu đố khác (Rất phổ biến):**
    Có thể bạn đang nhớ đến câu đố: *"Con gì có thể biến từ nam thành nữ?"*
    Câu trả lời là **Con Rắn**.
    *Giải thích:* Đây là một câu đố mẹo về sinh học của loài rắn (một số loài rắn cái có thể thay đổi giới tính).

**Kết luận:** Trong tiếng Việt, không có chữ nào chỉ mang ý nghĩa "nam" mà khi thêm dấu vào sẽ biến thành từ chỉ "nữ". Vì vậy, nếu ai đó hỏi bạn câu này, khả năng cao là họ đang muốn đố mẹo bạn bằng đáp án **"Chữ Nữ"**.


In [ ]:
chat("gemma4:e4b", [
    {"role": "system", "content": "You are a chef"},
    {"role": "user", "content": "hôm nay ăn gì?"},
], stream=True)

In [11]:
chat("gemma4:31b-cloud", [
    {"role": "system", "content": "You are a chef"},
    {"role": "user", "content": "hôm nay ăn gì?"},
], stream=True)

200
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.581766465Z","message":{"role":"assistant","content":"Ch\xc3\xa0o"},"done":false}'
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.622524909Z","message":{"role":"assistant","content":" b\xe1\xba\xa1n! V\xe1\xbb\x9bi t\xc6\xb0 c\xc3\xa1ch"},"done":false}'
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.663291843Z","message":{"role":"assistant","content":" l\xc3\xa0 m\xe1\xbb\x99t \xc4\x91\xe1\xba\xa7u"},"done":false}'
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.704060657Z","message":{"role":"assistant","content":" b\xe1\xba\xbfp, m\xc3\xacnh s\xe1\xba\xbd g\xe1\xbb\xa3i"},"done":false}'
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.704112247Z","message":{"role":"assistant","content":" \xc3\xbd cho b\xe1\xba\xa1n th\xe1\xbb\xb1c"},"done":false}'
b'{"model":"gemma4:31b","created_at":"2026-07-28T06:42:31.744810711Z","message":{"role":"assistant","content":" \xc4\x91\xc6\xa1n t\xc3\xb9

In [2]:
chat("gemma4:26b-gpu-90k", [
    {"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
    {"role": "user", "content": "What is the capital of France?"},
])

200
Ahoy there, matey! That landlubber city ye be searchin' for is none other than Paris! Arrr!


## Generate

In [3]:
r = requests.post(f"{WARP_URL}/api/generate", json={
    "model": "gemma4:31b-cloud",
    "messages": [
        {"role": "system", "content": "You are a pirate. Answer every question like a pirate."},
        {"role": "user", "content": "What is the capital of France?"}
    ],
    "stream": False,
})
print(r.status_code)
if r.ok:
    print(r.json()["response"])

200
Ahoy there, matey! That be Paris, the city o' lights! A fine place to plunder for art and fancy pastries, though I prefer the scent o' salt spray to the smell o' perfume! Savvy?


## Streaming chat

In [34]:
r = requests.post(f"{WARP_URL}/api/chat", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "count to 3"}],
    "stream": True,
}, stream=True)
if r.ok:
    for line in r.iter_lines():
        if line:
            print(line)

b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.4326544Z","message":{"role":"assistant","content":"1"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.5253613Z","message":{"role":"assistant","content":","},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.6142675Z","message":{"role":"assistant","content":" "},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.7073405Z","message":{"role":"assistant","content":"2"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.8021142Z","message":{"role":"assistant","content":","},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:13.9167147Z","message":{"role":"assistant","content":" "},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:14.0296252Z","message":{"role":"assistant","content":"3"},"done":false}'
b'{"model":"gemma4:e4b","created_at":"2026-07-14T06:39:14.1526181Z","message":{"role":"assistant","conte

## Embeddings

In [64]:
r = requests.post(f"{WARP_URL}/api/embed", json={
    "model": "embeddinggemma:300m",
    "input": "hello world",
})
print(r.status_code)
if r.ok:
    print(len(r.json()["embeddings"][0]), "dimensions")

200
768 dimensions


## OpenAI-compatible endpoint

Using `/v1/chat/completions` with OpenAI-compatible format. Routing works the same way (model suffix or `?source=` param).

In [ ]:
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    data = r.json()
    print(data["choices"][0]["message"]["content"])

In [9]:
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:31b-cloud",
    "messages": [{"role": "system", "content": "You are a pirate."},
                 {"role": "user", "content": "hello"}],
    "stream": False,
})
print(r.status_code)
if r.ok:
    data = r.json()
    print(data["choices"][0]["message"]["content"])

200
Ahoy there, ye scurvy dog! 🏴‍☠️

Welcome aboard! Ye look like ye've come from a faraway shore. Speak plain: be ye seekin' buried treasure, a flagon o' rum, or be ye just lookin' to sail the seven seas with a crew of salty sea-dogs?

State yer business, or it's the plank for ye! Har har!


In [8]:
# Streaming via /v1/chat/completions (SSE format)
r = requests.post(f"{WARP_URL}/v1/chat/completions", json={
    "model": "gemma4:e4b",
    "messages": [{"role": "user", "content": "count to 3"}],
    "stream": True,
}, stream=True)
print(r.status_code)
if r.ok:
    for line in r.iter_lines():
        if line:
            decoded = line.decode() if isinstance(line, bytes) else line
            if decoded.startswith("data: ") and decoded != "data: [DONE]":
                import json
                chunk = json.loads(decoded[6:])
                content = chunk["choices"][0]["delta"].get("content", "")
                if content:
                    print(content, end="", flush=True)
    print()

200
1, 2, 3
